# ==============================================================================
# CSIDA — Stage 1 comparison: paper's claim vs our WiDA-KAN reproduction vs tcn_pooled
# ==============================================================================
# নিচের প্রতিটি সেকশন (# %% CELL X) কপি করে Kaggle নোটবুকের আলাদা আলাদা সেলে পেস্ট
# করে রান করুন।
#
# CSIDA (Mendeley Data, DOI 10.17632/gyr6c4nbsc.1, https://data.mendeley.com/datasets/gyr6c4nbsc/1):
# Intel 5300 NIC দিয়ে collected CSI, 6 activities (pull left/right, lift up, press
# down, draw circle, draw zigzag), 5 users, 2 environments (office: 3 locations,
# classroom: 2 locations -> মোট 5 location), 3000 trials (6x5x5x20)।
#
# ডেটা **Zarr** format-এ (আপনার স্ক্রিনশট অনুযায়ী): csi_data_amp / csi_label_act /
# csi_label_env / csi_label_loc / csi_label_user, প্রতিটা একটা zarr array, .zgroup
# ফাইল দিয়ে চেনা যায় এটা zarr v2 ফরম্যাট।
#
# *** গুরুত্বপূর্ণ সততার নোট ***: আমি নিজে এই dataset ডাউনলোড করে দেখতে পারিনি
# (data.mendeley.com এই environment থেকে ব্লকড)। তাই Cell 1 (explore) টা **প্রথমে
# একবার একলা চালিয়ে** array shape/dtype/label range প্রিন্ট করা দেখে নিশ্চিত হও যে
# Cell 2-এর ধারণাগুলো (কোন axis time, কয়টা channel) ঠিক আছে কিনা -- ভুল হলে Cell
# 2-এর কমেন্টে বলা জায়গাটা বদলে দাও। বাকি সব (Stage 1/2/3, training loop, AMP,
# chunked preprocessing) WiMANS/ARIL script গুলোর থেকে হুবহু আনা এবং সেগুলোতে আগে
# verify করা প্যাটার্ন, শুধু data-loading অংশটাই নতুন এবং untested against real data।
#
# ৩-way comparison শেষে (Cell 10):
#   1. Paper's claim   -- WiDA-KAN পেপারের নিজস্ব CSIDA ablation থেকে (hardcoded reference)
#   2. Our resnet_attn  -- আমাদের নিজেদের বানানো WiDA-KAN reproduction (Stage1=original ResNet+gate)
#   3. Our tcn_pooled    -- Stage 1 পাল্টে lightweight dilated-TCN+pooling (ARIL-এ validated)
#
# *** amp+phase আপডেট (2026-08-26) ***: amp-only রান হয়ে গেছে (resnet_attn: HAR
# 72.06%, Loc 99.65%, paper's amp-only claim 76.98%/100.00% এর কাছাকাছি কিন্তু
# activity-তে -4.92pt পিছিয়ে)। এখন `INPUT_MODALITY = "amp_phase"` (Cell 0) সেট
# করে amp+phase দিয়ে দেখা হচ্ছে আমরা paper-এর amp+phase claim (96.84%/100.00%)
# এর কতটা কাছে যেতে পারি। Cell 1 এখন `PHASE_ARRAY_CANDIDATES`-এর মধ্যে কোনো
# phase array খুঁজে বের করে (csi_data_pha ইত্যাদি নামে) -- পেলে Cell 2 সেটা লোড
# করে **unwrap করে** (raw phase wrapped থাকে ±π এর মধ্যে, unwrap না করলে filter-এ
# artificial jump artifact তৈরি হয় -- এই bug টা আগে WiMANS-এ পাওয়া গিয়েছিল,
# results/wimans_amp_vs_amp_phase.md দ্রষ্টব্য) তারপর amp-এর সাথে channel-axis এ
# concatenate করে। phase array না পেলে (dataset-এ সত্যিই না থাকলে) স্বয়ংক্রিয়ভাবে
# amp-only তে fall back করবে, একটা warning প্রিন্ট করে -- crash করবে না। Cell 10
# এখন dynamically সঠিক paper-reference সারি বেছে নেয় (amp_only vs amp_phase)
# যেটার সাথে আসলে তুলনা করা হচ্ছে তার উপর ভিত্তি করে।

In [ ]:
# ============================================================
# 0 | Config
# ============================================================
import os, math, random, time, numpy as np, pandas as pd, torch, torch.nn as nn

SEED = 42

# Auto-find the CSIDA zarr root under /kaggle/input (looks for the csi_label_act
# array folder as the marker, same auto-discovery convention used in the WiMANS/
# ARIL scripts in this repo). Confirmed real path on Kaggle:
# /kaggle/input/datasets/mishatmilon/csida-dataset
DATASET_PATH = "/kaggle/input/datasets/mishatmilon/csida-dataset"
if not os.path.isdir(os.path.join(DATASET_PATH, "csi_label_act")):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "csi_label_act" in dirs:
            DATASET_PATH = root
            break
print("Using DATASET_PATH:", DATASET_PATH)

CACHE_DIR = "/kaggle/working/csida_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# Round 3 (DECIMATE=16/dropout=0.6/label_smoothing=0.1) reached activity 73.81%
# (tcn_pooled) / 74.69% (resnet_attn) vs paper's amp-only 76.98% -- within ~2-3pts,
# a credible reproduction (results/csida_amp_only_fixed_result.md). BUT the
# confusion matrix (results/csida_confusion_matrix_analysis.md) shows the SAME
# confusion pairs in both backbones (esp. classes {1,3,4}->5 and 0<->2<->3) --
# strong evidence the residual error is genuine class similarity / lost temporal
# detail, not an architecture limit. Working hypothesis: DECIMATE=16 (1800->113
# samples) may be throwing away exactly the fine temporal/frequency detail that
# separates similar continuous gestures (e.g. draw-circle vs draw-zigzag).
# Round 4 added mixup + time-masking (below), which regularize WITHOUT destroying
# temporal resolution -- so rolling DECIMATE back down to let them carry more of
# the anti-overfitting load, hoping to recover some of that resolution. This is a
# hypothesis, not yet confirmed on the real dataset -- verified only against
# synthetic data before shipping, same as every round so far.
DECIMATE      = 8           # was 16 (Round 3) -- see confusion-matrix analysis above
PREPROC_CHUNK = 300

BACKBONE_CH   = [64, 128, 256, 256]
BLOCKS        = [1, 1, 1, 1]
SHARED_DIM    = 256
WAVELET       = "dog"
DROPOUT       = 0.6        # was 0.5 -- pushing regularization further
SIGMA_FLOOR   = -1.0

# Label smoothing on both task heads' cross-entropy -- cheap, standard
# overfitting-targeted regularizer (see docs/CSIDA discussion 2026-08-26 on the
# "Hybrid Multi-Task Loss" proposal: this piece was judged sound and worth
# trying; the proposed AWL rewrite was NOT adopted -- it's mathematically
# unbounded below, see that discussion for the derivative proof).
LABEL_SMOOTHING = 0.1

# Train-time augmentation (activity-overfitting-targeted, see DECIMATE note above):
# small random time-shift + additive noise on the TRAIN split only (Cell 7).
AUGMENT_TRAIN     = True
AUG_MAX_SHIFT_FRAC = 0.10   # up to +/-10% of TIME_LEN as a circular time shift
AUG_NOISE_STD      = 0.05   # additive Gaussian noise, in units of the per-channel z-scored signal

# Round 4 (2026-08-27): TRAIN activity still ~99.9% vs VAL peak ~76% after Round 3
# (see results/csida_amp_only_fixed_result.md) -- pushing further with two more
# STANDARD, well-established regularizers not yet tried (not isolated one-at-a-time,
# same bundling discipline as Rounds 2/3):
#   1. Mixup (Zhang et al. 2017, "mixup: Beyond Empirical Risk Minimization") --
#      this is the mathematically SOUND version of the "CSIMix" idea from the
#      user's Hybrid Multi-Task Loss proposal: linearly interpolate two random
#      training samples' inputs AND both label sets with the same lambda ~
#      Beta(alpha, alpha), so the model sees convex combinations instead of only
#      exact training points -- proven general-purpose overfitting reducer,
#      unlike the proposed AWL rewrite which was rejected as unbounded below.
#   2. SpecAugment-style time-masking -- randomly zero out a few short time
#      segments per sample (train only), forcing the model to not rely on any
#      single time window. Applied in CSIDataset alongside the existing
#      shift+noise augmentation (Cell 7).
MIXUP_ALPHA     = 0.2   # 0 disables; Beta(alpha,alpha) mixing coefficient
TIME_MASK_FRAC  = 0.15  # total fraction of TIME_LEN masked out, train only
TIME_MASK_NUM   = 2     # split into this many masked segments

EPOCHS        = 120        # was 150; VAL peaked well before that in the overfitting run, no need to train past it
BATCH_SIZE    = 16
LR            = 0.001
LR_STEP       = 30
LR_GAMMA      = 0.5
WEIGHT_DECAY  = 5e-4       # raised from 1e-4 -- overfitting-targeted
GRAD_CLIP     = 5.0
VAL_FRACTION  = 0.15

FEATURE_EXTRACTORS = ["resnet_attn", "tcn_pooled"]   # our reproduction, then the ARIL-validated lightweight candidate
# Multi-seed noise-floor check (2026-08-27): Round 5's single-seed (42) numbers
# include a "beats the paper" claim (resnet_attn) and a widened resnet_attn-vs-
# tcn_pooled gap -- this project's established rigor discipline (see the ARIL
# 3-seed correction) says exactly this kind of claim needs mean+/-std across a
# few seeds before it's trusted. Same 3 seeds as ARIL's sweep for consistency.
SEED_LIST = [42, 123, 7]

# amp-only already run (results/... 72.06% har vs paper's 76.98%). Trying amp+phase
# now to see how close we get to the paper's amp+phase claim (96.84%/100.00%).
# Falls back to "amp" automatically (with a printed warning) if Cell 1 can't find a
# phase array in the zarr group -- doesn't hard-fail.
INPUT_MODALITY = "amp"     # "amp" | "amp_phase" -- re-confirm the overfitting fixes on amp-only first
PHASE_ARRAY_CANDIDATES = ["csi_data_pha", "csi_data_phase", "csi_data_phi", "csi_data_phase_amp"]

# WiDA-KAN paper's own CSIDA modality ablation (docs/Research_Gap_Analysis.md, Gap 4) --
# NOT re-measured here, just the reference to compare against.
PAPER_REFERENCE = {
    "amp_only":   {"activity": 76.98, "location": 100.00},
    "phase_only": {"activity": 84.10, "location": 98.30},
    "amp_phase":  {"activity": 96.84, "location": 100.00},
}

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
USE_AMP = (DEVICE.type == "cuda")
print("Device:", DEVICE, "| AMP:", USE_AMP)
print("Feature extractors to train:", FEATURE_EXTRACTORS, "| seeds:", SEED_LIST)
all_results = []      # collected across Cells 8-9, combined in Cell 10
trained_models = {}   # (feature_extractor, seed) -> best-checkpoint model, filled in Cell 8-9, used by Cell 11/12

In [ ]:
# ============================================================
# 1 | Explore the zarr structure BEFORE assuming anything about it
# ============================================================
# রান করে আগে দেখে নাও: (a) csi_data_amp-এর shape ক'টা axis, কোনটা time, (b) প্রতিটা
# label array-র unique value কয়টা (এখান থেকেই real NUM_ACT/NUM_ENV/NUM_LOC/NUM_USER
# বসবে, hardcode করা হয়নি ইচ্ছা করেই)।
#
# Kaggle-এ zarr প্রি-ইনস্টল থাকে না -- আগে install করে নিতে হবে (ModuleNotFoundError
# এড়াতে)। এই স্ক্রিনশট থেকে ইতিমধ্যে যা জানা গেছে (csi_label_env-এর .zarray থেকে):
# N=2844 samples (paper-described 3000 না, সম্ভবত কিছু ট্রায়াল বাদ পড়েছে এই
# uploaded ভার্সনে), csi_data_amp একটা 4D array (chunk file নাম "<i>.0.0.0" প্যাটার্ন
# দেখাচ্ছে axis-0 প্রতি-স্যাম্পল আলাদা chunk, বাকি ৩টা axis single-chunk) -- কিন্তু
# ওই ৩টা axis-এর আসল সাইজ (antenna/subcarrier/time) এখনো অজানা, নিচের প্রিন্ট থেকেই
# বেরিয়ে আসবে।
import subprocess, sys
try:
    import zarr
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "zarr"], check=True)
    import zarr
print("zarr version:", zarr.__version__)

root = zarr.open_group(DATASET_PATH, mode="r")
print("Arrays found in the zarr group:", list(root.array_keys()))
print()
for name in root.array_keys():
    arr = root[name]
    print(f"  {name:20s} shape={arr.shape}  dtype={arr.dtype}")

print("\n--- Label ranges (determines NUM_ACT / NUM_ENV / NUM_LOC / NUM_USER) ---")
for label_name in ["csi_label_act", "csi_label_env", "csi_label_loc", "csi_label_user"]:
    if label_name in root:
        vals = np.asarray(root[label_name][:]).reshape(-1)
        uniq = np.unique(vals)
        print(f"  {label_name:20s} unique values ({len(uniq)}): {uniq}")

# Look for a phase array under any of the candidate names -- needed for
# INPUT_MODALITY="amp_phase". Doesn't fail if none found; Cell 2 falls back to
# amp-only and prints a clear warning instead.
PHASE_ARRAY_NAME = next((n for n in PHASE_ARRAY_CANDIDATES if n in root.array_keys()), None)
if PHASE_ARRAY_NAME:
    print(f"\nPhase array found: '{PHASE_ARRAY_NAME}' shape={root[PHASE_ARRAY_NAME].shape} "
          f"dtype={root[PHASE_ARRAY_NAME].dtype} -- amp_phase mode is possible.")
else:
    print(f"\nNo phase array found among {PHASE_ARRAY_CANDIDATES} in this zarr group.")
    print("If INPUT_MODALITY='amp_phase' in Cell 0, Cell 2 will fall back to amp-only")
    print("and print a warning -- check the dataset's root folder for a phase array")
    print("under a different name if you need the amp+phase comparison.")

print("\nCHECK before continuing to Cell 2 (both already confirmed as real bugs, now fixed):")
print("  - csi_data_amp shape printed above -- Cell 2 auto-detects TIME as the LARGEST")
print("    axis among 1..end (confirmed correct for the real CSIDA shape (N,1800,3,114):")
print("    1800 is time, 114 is NOT). If some other dataset's largest axis genuinely")
print("    isn't time, override move_time_axis_last()'s heuristic in Cell 2.")
print("  - csi_label_loc alone shows only 3 unique values (not 5) because CSIDA numbers")
print("    locations PER-ENVIRONMENT (office 0,1,2 / classroom 0,1, reusing integers).")
print("    Cell 2 now combines (env, loc) into a genuine 5-way global location id.")

In [ ]:
# ============================================================
# 2 | Load CSI (+ phase if requested/available) + all 4 label arrays into memory
# ============================================================
amp_raw = np.asarray(root["csi_data_amp"][:], dtype=np.float32)     # shape: (N, ...) -- TIME axis auto-detected below
act_all_raw  = np.asarray(root["csi_label_act"][:]).reshape(-1).astype(np.int64)
env_all_raw  = np.asarray(root["csi_label_env"][:]).reshape(-1).astype(np.int64)
loc_all_raw  = np.asarray(root["csi_label_loc"][:]).reshape(-1).astype(np.int64)
user_all_raw = np.asarray(root["csi_label_user"][:]).reshape(-1).astype(np.int64)

def move_time_axis_last(x, label):
    """Auto-detect the TIME axis as the largest axis among 1..end (packet/time
    count is normally far bigger than antenna/subcarrier counts) and move it to
    the end. FIXES A REAL BUG found from the actual CSIDA shape (N, 1800, 3, 114):
    the old code always assumed the LAST axis was time (114, actually subcarriers)
    and flattened (1800 x 3) into ~5400 "channels" instead -- this silently
    destroyed the real temporal signal (explains the outsized 3.6M param count
    and the disproportionately low activity accuracy vs. location)."""
    non_sample_axes = list(range(1, x.ndim))
    time_axis = max(non_sample_axes, key=lambda ax: x.shape[ax])
    if time_axis != x.ndim - 1:
        print(f"  [{label}] detected TIME axis = {time_axis} (size {x.shape[time_axis]}), "
              f"moving to the end (was treating axis {x.ndim-1}, size {x.shape[-1]}, as time).")
        x = np.moveaxis(x, time_axis, -1)
    return x

amp_raw = move_time_axis_last(amp_raw, "amp")

N = amp_raw.shape[0]
TIME_LEN_RAW = amp_raw.shape[-1]
IN_CHANNELS_AMP = int(np.prod(amp_raw.shape[1:-1]))    # every axis EXCEPT sample and time, flattened
amp_flat = amp_raw.reshape(N, IN_CHANNELS_AMP, TIME_LEN_RAW)

# CSIDA's location labels turned out to be PER-ENVIRONMENT, not global -- confirmed
# from the real data: csi_label_loc showed only 3 unique values (office: 0,1,2 and
# classroom: 0,1, reusing the same small integers) instead of the expected 5. Build
# a genuine global location id by combining (environment, location) so office-loc-0
# and classroom-loc-0 are correctly treated as different physical places, not the
# same class. FIXES A REAL BUG: without this, the location task was an accidentally
# easier 3-way problem, not the paper's 5-way one -- probably why location accuracy
# looked unusually high.
combined_loc_key = np.array([f"{e}_{l}" for e, l in zip(env_all_raw, loc_all_raw)])

# Decide the ACTIVE modality now (falls back to "amp" if amp_phase was requested
# but Cell 1 didn't find a phase array -- never hard-fails).
ACTIVE_MODALITY = INPUT_MODALITY
if INPUT_MODALITY == "amp_phase" and PHASE_ARRAY_NAME is None:
    print("WARNING: INPUT_MODALITY='amp_phase' requested but no phase array was found "
          "in Cell 1 -- falling back to 'amp' only.")
    ACTIVE_MODALITY = "amp"

if ACTIVE_MODALITY == "amp_phase":
    pha_raw = np.asarray(root[PHASE_ARRAY_NAME][:], dtype=np.float32)
    pha_raw = move_time_axis_last(pha_raw, "phase")
    assert pha_raw.shape == amp_raw.shape, (
        f"amp shape {amp_raw.shape} != phase shape {pha_raw.shape} -- can't concatenate "
        "as-is; check whether phase needs its own axis handling.")
    IN_CHANNELS_PHA = int(np.prod(pha_raw.shape[1:-1]))
    pha_flat = pha_raw.reshape(N, IN_CHANNELS_PHA, TIME_LEN_RAW)
    # Unwrap phase along TIME before it ever reaches the lowpass filter in Cell 3 --
    # raw phase from commodity NICs is wrapped to (-pi, pi]; filtering wrapped phase
    # directly creates artificial jump artifacts (this exact bug was found and fixed
    # earlier in this project for WiMANS, see results/wimans_amp_vs_amp_phase.md).
    # unwrap() is a no-op if the phase is already continuous, so this is safe even if
    # CSIDA's shipped phase turns out to already be sanitized.
    pha_flat = np.unwrap(pha_flat, axis=-1).astype(np.float32)
    X_input_flat = np.concatenate([amp_flat, pha_flat], axis=1)   # (N, amp_ch + pha_ch, TIME_LEN_RAW)
    IN_CHANNELS = IN_CHANNELS_AMP + IN_CHANNELS_PHA
    print(f"amp_phase mode: {IN_CHANNELS_AMP} amp channels + {IN_CHANNELS_PHA} phase "
          f"channels (unwrapped) = {IN_CHANNELS} total")
else:
    X_input_flat = amp_flat
    IN_CHANNELS = IN_CHANNELS_AMP

# Re-index labels to a dense 0..K-1 range (defensive -- handles the case where raw
# label ids aren't already 0-indexed or contiguous).
def dense_reindex(labels):
    uniq = np.unique(labels)
    remap = {v: i for i, v in enumerate(uniq)}
    return np.array([remap[v] for v in labels], dtype=np.int64), len(uniq)

act_all, NUM_ACT   = dense_reindex(act_all_raw)
loc_all, NUM_LOC   = dense_reindex(combined_loc_key)     # (env, loc) combined -- see note above
env_all, NUM_ENV   = dense_reindex(env_all_raw)
user_all, NUM_USER = dense_reindex(user_all_raw)

print(f"Loaded: N={N}  ACTIVE_MODALITY={ACTIVE_MODALITY}  IN_CHANNELS={IN_CHANNELS}  TIME_LEN_RAW={TIME_LEN_RAW}")
print(f"NUM_ACT={NUM_ACT}  NUM_LOC={NUM_LOC}  NUM_ENV={NUM_ENV}  NUM_USER={NUM_USER}")
print("Expected from the dataset description: NUM_ACT=6, NUM_LOC=5, NUM_ENV=2, NUM_USER=5 --")
print("if these don't match, re-check Cell 1's printed shapes/label ranges before continuing.")

In [ ]:
# ============================================================
# 3 | Preprocess once, cache the whole pool (same chunked/vectorized pattern as
#     the WiMANS per-condition script -- one filtfilt() call per chunk, not per sample)
# ============================================================
from scipy.signal import butter, filtfilt

# NOTE: must match len(range(0, TIME_LEN_RAW, DECIMATE)) exactly, i.e. ceil(TIME_LEN_RAW/DECIMATE),
# NOT floor division -- TIME_LEN_RAW // DECIMATE silently mismatched the real length of
# X[:, :, ::DECIMATE] whenever DECIMATE doesn't divide TIME_LEN_RAW evenly (e.g. 1800/16=112.5,
# but [::16] actually produces 113 samples) -- a real bug caught here while testing DECIMATE=16
# (DECIMATE=8 never triggered it since 1800/8=225 divides evenly).
TIME_LEN = -(-TIME_LEN_RAW // DECIMATE)   # ceiling division
b_lp, a_lp = butter(4, 0.1, btype="low")
POOL_CACHE = os.path.join(CACHE_DIR, f"pool_{ACTIVE_MODALITY}_dec{DECIMATE}.npz")   # modality+DECIMATE in the name
# ^ IMPORTANT: without DECIMATE in the filename, changing DECIMATE and re-running
# would silently reload a stale cache built at the OLD decimation (a real bug hit
# while testing this fix) -- always encode every preprocessing knob that changes
# X_all's shape/content into the cache filename.

if os.path.exists(POOL_CACHE):
    print("Loading cached pool from", POOL_CACHE)
    d = np.load(POOL_CACHE)
    X_all = d["X"]
else:
    X_all = np.empty((N, IN_CHANNELS, TIME_LEN), dtype=np.float32)
    t0 = time.time()
    for start in range(0, N, PREPROC_CHUNK):
        end = min(start + PREPROC_CHUNK, N)
        chunk = X_input_flat[start:end]                           # (b, C, TIME_LEN_RAW)
        filtered = filtfilt(b_lp, a_lp, chunk, axis=-1)           # ONE vectorized call per chunk
        dec = filtered[:, :, ::DECIMATE].astype(np.float32)
        mean = dec.mean(axis=2, keepdims=True)
        std  = dec.std(axis=2, keepdims=True) + 1e-8
        X_all[start:end] = (dec - mean) / std
        print(f"  {end}/{N} preprocessed ({time.time()-t0:.1f}s)", flush=True)
    np.savez(POOL_CACHE, X=X_all)
    print(f"Cached -> {POOL_CACHE}  (shape {X_all.shape})")

print(f"Pool ready: X_all {X_all.shape}")

In [ ]:
# ============================================================
# 4 | Build train/val/test index masks
# ============================================================
from sklearn.model_selection import train_test_split

def get_split_indices(mode="random", held_out=None, seed=SEED):
    all_idx = np.arange(N)
    if mode == "random":
        trainval_idx, test_idx = train_test_split(
            all_idx, test_size=0.2, random_state=seed, stratify=act_all)
    elif mode == "cross_env":
        test_idx     = all_idx[env_all == held_out]
        trainval_idx = all_idx[env_all != held_out]
    else:
        raise ValueError(mode)
    train_idx, val_idx = train_test_split(
        trainval_idx, test_size=VAL_FRACTION, random_state=seed,
        stratify=act_all[trainval_idx])
    return train_idx, val_idx, test_idx

In [ ]:
# ============================================================
# 5 | Stage-1 feature extractors: resnet_attn (original) / tcn_pooled (ARIL-validated)
# ============================================================
class ResidualBlock1D(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.conv1 = nn.Conv1d(c_in, c_out, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm1d(c_out)
        self.relu  = nn.ReLU(inplace=True)
        self.short = (nn.Identity() if c_in == c_out else
                      nn.Sequential(nn.Conv1d(c_in, c_out, 1, bias=False), nn.BatchNorm1d(c_out)))
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + self.short(x))

class AttentionRefine1D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.gate = nn.Conv1d(channels, channels, 1)
    def forward(self, x):
        return x * torch.sigmoid(self.gate(x))

class ResNetAttnBackbone(nn.Module):
    """Original WiDA-KAN Stage 1 -- our own faithful reproduction, unchanged from
    the ARIL/WiMANS scripts in this repo."""
    def __init__(self, in_ch, stage_ch=BACKBONE_CH, blocks=BLOCKS):
        super().__init__()
        c0 = stage_ch[0]
        self.stem = nn.Sequential(
            nn.Conv1d(in_ch, c0, 7, padding=3, bias=False),
            nn.BatchNorm1d(c0), nn.ReLU(inplace=True), nn.MaxPool1d(2))
        self.stages = nn.ModuleList()
        prev = c0
        for ch, nb in zip(stage_ch, blocks):
            layers = [ResidualBlock1D(prev if i == 0 else ch, ch) for i in range(nb)]
            layers += [AttentionRefine1D(ch), nn.MaxPool1d(2)]
            self.stages.append(nn.Sequential(*layers)); prev = ch
        self.out_dim = prev
        self.gap = nn.AdaptiveAvgPool1d(1)
    def forward(self, x):
        x = self.stem(x)
        for s in self.stages:
            x = s(x)
        return self.gap(x).flatten(1)

class TCNBlock1D(nn.Module):
    def __init__(self, c_in, c_out, k=3, dilation=1):
        super().__init__()
        pad = (k - 1) * dilation // 2
        self.dw = nn.Conv1d(c_in, c_in, k, padding=pad, dilation=dilation, groups=c_in, bias=False)
        self.pw = nn.Conv1d(c_in, c_out, 1, bias=False)
        self.bn = nn.BatchNorm1d(c_out)
        self.act = nn.ReLU(inplace=True)
        self.short = (nn.Identity() if c_in == c_out else
                      nn.Sequential(nn.Conv1d(c_in, c_out, 1, bias=False), nn.BatchNorm1d(c_out)))
    def forward(self, x):
        out = self.act(self.bn(self.pw(self.dw(x))))
        return self.act(out + self.short(x))

class TCNBackbone(nn.Module):
    """ARIL-validated candidate (results/aril_stage1_lightweight_sweep_3seed.md):
    +1.4 avg points over resnet_attn at 30% of the parameters, pool=True cuts
    MACs ~72% with no accuracy cost there. First test of whether this transfers
    to CSIDA (WiMANS's partial result was mixed -- see
    results/wimans_tcn_per_condition_partial.md)."""
    def __init__(self, in_ch, channels=BACKBONE_CH, out_dim=SHARED_DIM, pool=True):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(in_ch, channels[0], 1, bias=False),
                                   nn.BatchNorm1d(channels[0]), nn.ReLU(inplace=True))
        blocks = []
        prev = channels[0]
        for i, ch in enumerate(channels):
            blocks.append(TCNBlock1D(prev, ch, k=3, dilation=2 ** i))
            if pool:
                blocks.append(nn.MaxPool1d(2))
            prev = ch
        self.blocks = nn.Sequential(*blocks)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.proj = nn.Identity() if prev == out_dim else nn.Linear(prev, out_dim)
        self.out_dim = out_dim
    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        return self.proj(self.gap(x).flatten(1))

def build_backbone(name):
    if name == "resnet_attn":
        return ResNetAttnBackbone(IN_CHANNELS)
    if name == "tcn_pooled":
        return TCNBackbone(IN_CHANNELS, pool=True)
    if name == "tcn":
        return TCNBackbone(IN_CHANNELS, pool=False)
    raise ValueError(f"Unknown feature extractor: {name}")

_x = torch.randn(2, IN_CHANNELS, TIME_LEN)
for _name in FEATURE_EXTRACTORS:
    _out = build_backbone(_name).eval()(_x)
    assert tuple(_out.shape) == (2, SHARED_DIM), f"{_name} broke the output contract: {_out.shape}"
    print(f"  {_name:12s} -> output shape {tuple(_out.shape)}  OK  (CSIDA input {tuple(_x.shape)})")
del _x, _out

In [ ]:
# ============================================================
# 6 | WavKANLayer + WiDAKAN(feature_extractor) + AdaptiveWeightedLoss
# ============================================================
class WavKANLayer(nn.Module):
    def __init__(self, in_dim, out_dim, wavelet="dog"):
        super().__init__()
        self.wavelet = wavelet
        self.weight      = nn.Parameter(torch.randn(out_dim, in_dim) / math.sqrt(in_dim))
        self.translation = nn.Parameter(torch.zeros(out_dim, in_dim))
        self.log_scale   = nn.Parameter(torch.zeros(out_dim, in_dim))
        self.bn = nn.BatchNorm1d(out_dim)
    def _psi(self, z):
        if self.wavelet == "dog":
            return -z * torch.exp(-0.5 * z ** 2)
        if self.wavelet == "mexican_hat":
            return (1 - z ** 2) * torch.exp(-0.5 * z ** 2)
        if self.wavelet == "shannon":
            return torch.sinc(z) * torch.cos(1.5 * math.pi * z)
        raise ValueError(self.wavelet)
    def forward(self, x):
        x = x.unsqueeze(1)
        z = (x - self.translation.unsqueeze(0)) * torch.exp(-self.log_scale).unsqueeze(0)
        y = (self._psi(z) * self.weight.unsqueeze(0)).sum(dim=-1)
        return self.bn(y)

class WiDAKAN(nn.Module):
    def __init__(self, feature_extractor):
        super().__init__()
        self.backbone = build_backbone(feature_extractor)
        self.feat_bn  = nn.BatchNorm1d(self.backbone.out_dim)
        self.wavkan   = WavKANLayer(self.backbone.out_dim, SHARED_DIM, WAVELET)
        self.dropout  = nn.Dropout(DROPOUT)
        self.har_head = nn.Linear(SHARED_DIM, NUM_ACT)
        self.loc_head = nn.Linear(SHARED_DIM, NUM_LOC)
    def forward(self, x):
        f = self.feat_bn(self.backbone(x))
        h = self.dropout(self.wavkan(f))
        return self.har_head(h), self.loc_head(h)

class AdaptiveWeightedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_sigma_har = nn.Parameter(torch.zeros(1))
        self.log_sigma_loc = nn.Parameter(torch.zeros(1))
    def forward(self, ce_har, ce_loc):
        ls_h = self.log_sigma_har.clamp(min=SIGMA_FLOOR)
        ls_l = self.log_sigma_loc.clamp(min=SIGMA_FLOOR)
        return (torch.exp(-2 * ls_h) * ce_har + ls_h
                + torch.exp(-2 * ls_l) * ce_loc + ls_l)

In [ ]:
# ============================================================
# 7 | make_loader / evaluate / run_experiment / run_condition
# ============================================================
from torch.utils.data import Dataset, DataLoader

class CSIDataset(Dataset):
    """TensorDataset-like wrapper that optionally applies light train-time
    augmentation (random circular time-shift + additive Gaussian noise) --
    added specifically to fight the overfitting seen on the real CSIDA run
    (results/csida_amp_only_fixed_result.md: TRAIN activity ~100% vs VAL
    ~48-58% by epoch 100). Never applied to val/test (augment=False there),
    so evaluation always sees the real, unmodified signal."""
    def __init__(self, X, act, loc, augment=False):
        self.X = torch.from_numpy(X)
        self.act = torch.from_numpy(act)
        self.loc = torch.from_numpy(loc)
        self.augment = augment
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, i):
        x = self.X[i]
        if self.augment:
            max_shift = max(1, int(x.shape[-1] * AUG_MAX_SHIFT_FRAC))
            shift = int(torch.randint(-max_shift, max_shift + 1, (1,)).item())
            if shift != 0:
                x = torch.roll(x, shifts=shift, dims=-1)
            if AUG_NOISE_STD > 0:
                x = x + torch.randn_like(x) * AUG_NOISE_STD
            if TIME_MASK_FRAC > 0:
                x = x.clone()
                T = x.shape[-1]
                seg_len = max(1, int(T * TIME_MASK_FRAC / TIME_MASK_NUM))
                for _ in range(TIME_MASK_NUM):
                    start = random.randint(0, max(0, T - seg_len))
                    x[:, start:start + seg_len] = 0.0
        return x, self.act[i], self.loc[i]

def make_loader(idx, shuffle, augment=False):
    ds = CSIDataset(X_all[idx], act_all[idx], loc_all[idx], augment=augment)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, drop_last=shuffle,
                       num_workers=2, pin_memory=True, persistent_workers=shuffle)

def mixup_batch(x, act, loc, alpha):
    """Standard mixup (Zhang et al. 2017): mix a batch with a random shuffle of
    itself, same lambda for inputs and both label sets. alpha<=0 disables (returns
    lam=1.0, i.e. an ordinary batch, no shuffle needed)."""
    if alpha <= 0:
        return x, act, loc, act, loc, 1.0
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(x.size(0), device=x.device)
    x_mixed = lam * x + (1 - lam) * x[perm]
    return x_mixed, act, loc, act[perm], loc[perm], lam

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ch = cl = tot = 0
    for xb, ab, lb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        ab = ab.to(DEVICE, non_blocking=True)
        lb = lb.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            oh, ol = model(xb)
        ch += (oh.argmax(1) == ab).sum().item()
        cl += (ol.argmax(1) == lb).sum().item()
        tot += xb.size(0)
    return 100 * ch / tot, 100 * cl / tot

def run_experiment(feature_extractor, seed=SEED, epochs=EPOCHS, print_every=10):
    set_seed(seed)
    train_idx, val_idx, test_idx = get_split_indices("random", seed=seed)
    train_loader = make_loader(train_idx, shuffle=True, augment=AUGMENT_TRAIN)
    val_loader   = make_loader(val_idx,   shuffle=False)                        # never augmented
    test_loader  = make_loader(test_idx,  shuffle=False)                        # never augmented

    model = WiDAKAN(feature_extractor).to(DEVICE)
    awl   = AdaptiveWeightedLoss().to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    params = list(model.parameters()) + list(awl.parameters())
    optimizer = torch.optim.AdamW(params, lr=LR, betas=(0.9, 0.999), weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP, gamma=LR_GAMMA)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)

    tag = f"{feature_extractor}|seed{seed}"
    print(f"\n--- {tag} --- n_train={len(train_idx)} n_val={len(val_idx)} n_test={len(test_idx)}")

    best_val_avg, best_state, best_val_metrics = -1.0, None, None
    n_params = sum(p.numel() for p in model.parameters()) + sum(p.numel() for p in awl.parameters())
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        ep_t0 = time.time()
        model.train()
        run_loss = ch = cl = tot = 0.0
        for xb, ab, lb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            ab = ab.to(DEVICE, non_blocking=True)
            lb = lb.to(DEVICE, non_blocking=True)
            xb, ab_a, lb_a, ab_b, lb_b, lam = mixup_batch(xb, ab, lb, MIXUP_ALPHA)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                oh, ol = model(xb)
                loss_h = lam * criterion(oh, ab_a) + (1 - lam) * criterion(oh, ab_b)
                loss_l = lam * criterion(ol, lb_a) + (1 - lam) * criterion(ol, lb_b)
                loss = awl(loss_h, loss_l)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            bs = xb.size(0); run_loss += loss.item() * bs; tot += bs
            # mixup makes "train accuracy" a soft/approximate quantity (blend of the
            # two mixed samples' correctness) -- fine for the printed diagnostic,
            # test/val accuracy (never mixed) remains the real number that matters.
            ch += (lam * (oh.argmax(1) == ab_a).float() + (1 - lam) * (oh.argmax(1) == ab_b).float()).sum().item()
            cl += (lam * (ol.argmax(1) == lb_a).float() + (1 - lam) * (ol.argmax(1) == lb_b).float()).sum().item()
        scheduler.step()

        tr_har, tr_loc = 100 * ch / tot, 100 * cl / tot
        val_har, val_loc = evaluate(model, val_loader)
        val_avg = (val_har + val_loc) / 2
        if val_avg > best_val_avg:
            best_val_avg = val_avg
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_val_metrics = {"epoch": epoch, "val_har": val_har, "val_loc": val_loc}

        if epoch % print_every == 0 or epoch == 1 or epoch == epochs:
            print(f"  ep {epoch:3d}/{epochs} | {time.time()-ep_t0:4.1f}s | loss {run_loss/tot:6.3f} | "
                  f"TRAIN har {tr_har:5.1f} loc {tr_loc:5.1f} | VAL har {val_har:5.1f} loc {val_loc:5.1f}",
                  flush=True)

    model.load_state_dict(best_state)
    test_har, test_loc = evaluate(model, test_loader)
    result = {
        "feature_extractor": feature_extractor, "modality": ACTIVE_MODALITY, "seed": seed,
        "total_params_M": n_params / 1e6, "best_epoch": best_val_metrics["epoch"],
        "test_har": test_har, "test_loc": test_loc, "test_avg": (test_har + test_loc) / 2,
    }
    print(f"[{tag}] DONE in {time.time()-t0:5.1f}s -> TEST har {test_har:.2f} | loc {test_loc:.2f} "
          f"| avg {(test_har+test_loc)/2:.2f} (best ep {best_val_metrics['epoch']} by VAL, "
          f"params {n_params/1e6:.3f}M)")
    # Keyed by (feature_extractor, seed), NOT just feature_extractor -- with
    # multiple seeds in SEED_LIST, each seed has its OWN train/val/test split
    # (get_split_indices uses seed as the split's random_state too), so a
    # model trained under seed=7 must never be evaluated against seed=42's
    # "test" split in Cell 11/12 -- some of those "test" samples could have
    # been in seed=7's train set. Cell 11/12 select trained_models[(fe, SEED)]
    # specifically to keep the diagnostic model and its split consistent.
    trained_models[(feature_extractor, seed)] = model
    return result

In [ ]:
# ============================================================
# 8 | Train resnet_attn
# ============================================================
for _seed in SEED_LIST:
    all_results.append(run_experiment("resnet_attn", seed=_seed))

In [ ]:
# ============================================================
# 9 | Train tcn_pooled
# ============================================================
for _seed in SEED_LIST:
    all_results.append(run_experiment("tcn_pooled", seed=_seed))

In [ ]:
# ============================================================
# 10 | Combine into the 3-way comparison table (modality-aware)
# ============================================================
results_df = pd.DataFrame(all_results)
print("=== Our runs (this notebook, one row per seed) ===")
print(results_df[["feature_extractor", "modality", "seed", "total_params_M",
                   "test_har", "test_loc", "test_avg"]].to_string(index=False))

results_df.to_csv(os.path.join(CACHE_DIR, f"csida_results_{ACTIVE_MODALITY}.csv"), index=False)
results_df.to_csv(f"csida_results_{ACTIVE_MODALITY}.csv", index=False)

# Aggregate mean +/- std across seeds per backbone (SEED_LIST may have 1 or
# several seeds -- with 1 seed, std is NaN/undefined, printed as 0.00 below.
# This replaces an earlier version of this cell that used results_df.iloc[0]
# per backbone, which silently picked an arbitrary single seed once
# SEED_LIST grew past length 1 -- a real bug caught while adding the
# multi-seed noise-floor check).
summary = results_df.groupby("feature_extractor").agg(
    total_params_M=("total_params_M", "first"),
    har_mean=("test_har", "mean"), har_std=("test_har", "std"),
    loc_mean=("test_loc", "mean"), loc_std=("test_loc", "std"),
    avg_mean=("test_avg", "mean"), avg_std=("test_avg", "std"),
    n_seeds=("seed", "count"),
).reindex(FEATURE_EXTRACTORS)
summary[["har_std", "loc_std", "avg_std"]] = summary[["har_std", "loc_std", "avg_std"]].fillna(0.0)
print(f"\n=== Mean +/- std across {results_df['seed'].nunique()} seed(s) ===")
print(summary.to_string(float_format=lambda v: f"{v:.2f}"))
summary.to_csv(os.path.join(CACHE_DIR, f"csida_summary_{ACTIVE_MODALITY}.csv"))
summary.to_csv(f"csida_summary_{ACTIVE_MODALITY}.csv")

# Pick the paper reference row matching what we actually ran, per modality --
# this is what fixes the earlier amp-only run's comparison automatically switching
# to amp+phase's row (96.84%/100.00%) once ACTIVE_MODALITY == "amp_phase".
paper_key = "amp_phase" if ACTIVE_MODALITY == "amp_phase" else "amp_only"
paper_ref = PAPER_REFERENCE[paper_key]

print(f"\n=== 3-WAY COMPARISON (modality: {ACTIVE_MODALITY}, mean across seeds) ===")
print(f"{'Source':32s} {'Activity':>14s} {'Location':>14s}")
print(f"{'-'*32} {'-'*14} {'-'*14}")
print(f"{'Paper claim (' + paper_key + ')':32s} {paper_ref['activity']:14.2f} {paper_ref['location']:14.2f}   <- fair comparison for this run")
for other_key, other_ref in PAPER_REFERENCE.items():
    if other_key != paper_key:
        print(f"{'Paper claim (' + other_key + ')':32s} {other_ref['activity']:14.2f} {other_ref['location']:14.2f}   <- NOT this run's comparison")
for fe, row in summary.iterrows():
    har_str = f"{row['har_mean']:.2f}+/-{row['har_std']:.2f}"
    loc_str = f"{row['loc_mean']:.2f}+/-{row['loc_std']:.2f}"
    print(f"{'Ours: ' + fe + ' (' + ACTIVE_MODALITY + ')':32s} {har_str:>14s} {loc_str:>14s}   "
          f"(params {row['total_params_M']:.3f}M, n={int(row['n_seeds'])})")

print(f"\n=== point change vs. paper's {paper_key} claim (using mean across seeds) ===")
for fe, row in summary.iterrows():
    d_act = row["har_mean"] - paper_ref["activity"]
    d_loc = row["loc_mean"] - paper_ref["location"]
    print(f"  {fe:12s} activity {d_act:+.2f} pts | location {d_loc:+.2f} pts")

if "resnet_attn" in summary.index and "tcn_pooled" in summary.index:
    ra = summary.loc["resnet_attn"]; tp = summary.loc["tcn_pooled"]
    print(f"\n=== tcn_pooled vs our own resnet_attn reproduction (mean across seeds) ===")
    print(f"  activity: {tp['har_mean'] - ra['har_mean']:+.2f} pts | "
          f"location: {tp['loc_mean'] - ra['loc_mean']:+.2f} pts | "
          f"params: {tp['total_params_M']/ra['total_params_M']*100:.1f}% of resnet_attn")

import matplotlib.pyplot as plt
labels = [f"Paper\n({paper_key})"] + [f"Ours:\n{fe}" for fe in summary.index]
har_vals = [paper_ref["activity"]] + summary["har_mean"].tolist()
loc_vals = [paper_ref["location"]] + summary["loc_mean"].tolist()
har_errs = [0] + summary["har_std"].tolist()
loc_errs = [0] + summary["loc_std"].tolist()
x = np.arange(len(labels)); w = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - w/2, har_vals, w, yerr=har_errs, capsize=4, label="Activity")
ax.bar(x + w/2, loc_vals, w, yerr=loc_errs, capsize=4, label="Location")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("Accuracy (%)")
ax.set_title(f"CSIDA ({ACTIVE_MODALITY}): paper's claim vs our resnet_attn vs tcn_pooled "
             f"(mean +/- std, n={results_df['seed'].nunique()} seed(s))")
ax.legend(); ax.grid(alpha=.3, axis="y")
plt.tight_layout(); plt.show()

print(f"\nSaved -> csida_results_{ACTIVE_MODALITY}.csv (per-seed), "
      f"csida_summary_{ACTIVE_MODALITY}.csv (mean+/-std). Download both from "
      f"Kaggle's Output tab and add them to results/ in the repo.")

In [ ]:
# ============================================================
# 11 | Per-class report + confusion matrix, reusing the trained models (no retraining)
# ============================================================
# NOTE: raw activity/location labels are just dense-reindexed integers (0..K-1) --
# we don't have a verified mapping back to the real gesture names (pull left/right,
# lift up, press down, draw circle, draw zigzag) or physical locations from the
# label arrays alone, so axes are shown by class index, not by guessed names.
from sklearn.metrics import confusion_matrix, classification_report

_, _, cm_test_idx = get_split_indices("random", seed=SEED)
cm_test_loader = make_loader(cm_test_idx, shuffle=False)   # same split run_experiment used, no augmentation

@torch.no_grad()
def get_predictions(model, loader):
    model.eval()
    all_act_true, all_act_pred, all_loc_true, all_loc_pred = [], [], [], []
    for xb, ab, lb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            oh, ol = model(xb)
        all_act_true.append(ab.numpy()); all_act_pred.append(oh.argmax(1).cpu().numpy())
        all_loc_true.append(lb.numpy()); all_loc_pred.append(ol.argmax(1).cpu().numpy())
    return (np.concatenate(all_act_true), np.concatenate(all_act_pred),
            np.concatenate(all_loc_true), np.concatenate(all_loc_pred))

ACT_LABELS = [str(i) for i in range(NUM_ACT)]
LOC_LABELS = [str(i) for i in range(NUM_LOC)]

# Only the seed=SEED model for each backbone -- matches cm_test_idx's split
# (built with seed=SEED above). With multiple seeds in SEED_LIST, every other
# seed's model was trained on a DIFFERENT split and must not be evaluated here.
for fe_name in FEATURE_EXTRACTORS:
    fe_model = trained_models.get((fe_name, SEED))
    if fe_model is None:
        print(f"[skip] no seed={SEED} model found for {fe_name} in trained_models")
        continue
    act_true, act_pred, loc_true, loc_pred = get_predictions(fe_model, cm_test_loader)

    print(f"\n{'='*70}\n{fe_name}: ACTIVITY per-class report\n{'='*70}")
    print(classification_report(act_true, act_pred, target_names=ACT_LABELS, zero_division=0))
    print(f"\n{fe_name}: LOCATION per-class report")
    print(classification_report(loc_true, loc_pred, target_names=LOC_LABELS, zero_division=0))

    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    cm_act = confusion_matrix(act_true, act_pred, labels=range(NUM_ACT))
    ax[0].imshow(cm_act, cmap="Blues")
    ax[0].set_xticks(range(NUM_ACT)); ax[0].set_yticks(range(NUM_ACT))
    ax[0].set_xticklabels(ACT_LABELS); ax[0].set_yticklabels(ACT_LABELS)
    ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("True")
    ax[0].set_title(f"{fe_name}: Activity confusion")
    for i in range(NUM_ACT):
        for j in range(NUM_ACT):
            ax[0].text(j, i, cm_act[i, j], ha="center", va="center",
                       color="white" if cm_act[i, j] > cm_act.max() / 2 else "black", fontsize=8)

    cm_loc = confusion_matrix(loc_true, loc_pred, labels=range(NUM_LOC))
    ax[1].imshow(cm_loc, cmap="Blues")
    ax[1].set_xticks(range(NUM_LOC)); ax[1].set_yticks(range(NUM_LOC))
    ax[1].set_xticklabels(LOC_LABELS); ax[1].set_yticklabels(LOC_LABELS)
    ax[1].set_xlabel("Predicted"); ax[1].set_ylabel("True")
    ax[1].set_title(f"{fe_name}: Location confusion")
    for i in range(NUM_LOC):
        for j in range(NUM_LOC):
            ax[1].text(j, i, cm_loc[i, j], ha="center", va="center",
                       color="white" if cm_loc[i, j] > cm_loc.max() / 2 else "black", fontsize=8)
    plt.suptitle(f"{fe_name} confusion matrices (test set, n={len(act_true)})")
    plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# 12 | Is class 5 a class-imbalance problem, or a genuine signal-similarity
#      problem? Three checks, all reusing data/models already in memory:
#      (a) per-class sample counts in the full dataset AND in each split
#          (train_test_split above already stratifies by act_all, so an
#          imbalance in the raw data would still show up proportionally in
#          every split -- this checks whether that imbalance exists at all);
#      (b) the average raw preprocessed waveform per activity class, to see
#          by eye whether the classes that get confused in Cell 11 (esp.
#          {1,3,4} <-> 5) actually look alike;
#      (c) a 2D PCA of the model's own shared embedding (the WavKAN output,
#          right before the two task heads) on the test set, colored by true
#          activity -- if classes overlap here even for a trained model, the
#          confusion is a genuine representation-level ambiguity, not a
#          fixable classifier-head issue.
# ============================================================

# --- (a) class balance ---------------------------------------------------
cm_train_idx, cm_val_idx, cm_test_idx = get_split_indices("random", seed=SEED)
print("=== Activity class balance ===")
print(f"{'class':>6} {'full N':>8} {'train':>8} {'val':>8} {'test':>8}")
full_counts  = np.bincount(act_all, minlength=NUM_ACT)
train_counts = np.bincount(act_all[cm_train_idx], minlength=NUM_ACT)
val_counts   = np.bincount(act_all[cm_val_idx],   minlength=NUM_ACT)
test_counts  = np.bincount(act_all[cm_test_idx],  minlength=NUM_ACT)
for c in range(NUM_ACT):
    print(f"{c:>6} {full_counts[c]:>8} {train_counts[c]:>8} {val_counts[c]:>8} {test_counts[c]:>8}")
imbalance_ratio = full_counts.max() / full_counts.min()
print(f"\nmax/min class-count ratio (full dataset): {imbalance_ratio:.2f}x "
      f"({'looks balanced, imbalance is NOT the driver of class 5 errors' if imbalance_ratio < 1.3 else 'notable imbalance -- consider class weighting'})")

# --- (b) average raw waveform per class -----------------------------------
fig, axes = plt.subplots(2, 3, figsize=(15, 6), sharey=True)
for c in range(NUM_ACT):
    ax = axes.flat[c]
    class_idx = np.where(act_all == c)[0]
    mean_wave = X_all[class_idx].mean(axis=(0, 1))    # mean over samples AND channels -> (TIME_LEN,)
    std_wave  = X_all[class_idx].mean(axis=1).std(axis=0)  # spread across samples, channel-averaged
    ax.plot(mean_wave, lw=1.5)
    ax.fill_between(range(len(mean_wave)), mean_wave - std_wave, mean_wave + std_wave, alpha=0.2)
    ax.set_title(f"class {c} (n={len(class_idx)})")
    ax.grid(alpha=.3)
plt.suptitle("Mean channel-averaged waveform per activity class (shaded = ±1 std across samples)")
plt.tight_layout(); plt.show()
print("If classes 1/3/4/5 (the confused group from Cell 11) trace very similar "
      "curves here, that is direct visual evidence of genuine signal similarity "
      "-- not something a bigger model or more epochs can fix on its own.")

# --- (c) PCA of the shared embedding, colored by true activity ------------
from sklearn.decomposition import PCA

@torch.no_grad()
def get_embeddings(model, loader):
    model.eval()
    feats, labels = [], []
    for xb, ab, lb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            f = model.feat_bn(model.backbone(xb))
            h = model.wavkan(f)
        feats.append(h.float().cpu().numpy()); labels.append(ab.numpy())
    return np.concatenate(feats), np.concatenate(labels)

# Same seed=SEED restriction as Cell 11 -- every model here must match
# cm_test_loader's split (built with seed=SEED).
_seed_models = [(fe, trained_models[(fe, SEED)]) for fe in FEATURE_EXTRACTORS if (fe, SEED) in trained_models]
if _seed_models:
    fig, axes = plt.subplots(1, len(_seed_models), figsize=(6.5 * len(_seed_models), 5.5), squeeze=False)
    for ax, (fe_name, fe_model) in zip(axes[0], _seed_models):
        emb, emb_labels = get_embeddings(fe_model, cm_test_loader)
        emb_2d = PCA(n_components=2, random_state=SEED).fit_transform(emb)
        sc = ax.scatter(emb_2d[:, 0], emb_2d[:, 1], c=emb_labels, cmap="tab10", s=14, alpha=0.8)
        ax.set_title(f"{fe_name}: shared-embedding PCA, colored by true activity")
        ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
        legend1 = ax.legend(*sc.legend_elements(), title="class", loc="best", fontsize=8)
        ax.add_artist(legend1)
    plt.tight_layout(); plt.show()
    print("If class 5's points form their own visible cluster here, the model IS "
          "separating it internally and the errors are closer to the decision "
          "boundary (worth trying: per-class loss weighting, or simply more "
          "training signal). If class 5's points are scattered inside classes "
          "1/3/4's clusters, the model genuinely cannot tell them apart from "
          "this input representation -- that points back to preprocessing "
          "(temporal resolution, DECIMATE) rather than the classifier head.")